# LOB Feature Engineering

Input: an existing pandas DataFrame named `modeling_df` containing the raw LOB/trade-flow columns.

Output: the **same `modeling_df`** with engineered feature columns added.

All three optional future targets share one configurable horizon `TARGET_HORIZON_MINUTES = H`:

- `mid_return_t_plus_H = mid[t+H] / mid[t] - 1`
- `target_log_return_Hm = log(mid[t+H] / mid[t])`
- `target_log_return_Hm_volnorm = target_log_return_Hm / past_vol_Hm`

With the default `H=30`, the names are exactly `mid_return_t_plus_30`, `target_log_return_30m`, and `target_log_return_30m_volnorm`. Target leads are computed within `(RIC, date_dt)` only.


In [ ]:
# Step 0 — imports
import pandas as pd

from feature_engineering_utils import add_lob_features


## Step 1 — configuration

`ROWS_PER_MINUTE` maps calendar minutes to rows. The target horizon is configured once and reused by all 3 optional targets.


In [ ]:
# Data frequency
# 1-minute rows -> 1; 5-second rows -> 12; 1-second rows -> 60
ROWS_PER_MINUTE = 1

# LOB structure
LEVELS = 8
GROUP_COLS = ("RIC", "date_dt")
TIME_COL = "bucketEnd"
DEPTH_KS = (1, 3, 5, 8)
ROLLING_WINDOWS = (3, 10, 30)
VOLATILITY_MINUTES = (5, 15, 30, 60)
DISTANCE_DECAY = 0.5

# ------------------------------------------------------------
# Optional future targets — one configurable horizon for all 3
# ------------------------------------------------------------
TARGET_HORIZON_MINUTES = 30

ADD_TARGET_MID_RETURN = True
ADD_TARGET_LOG_RETURN = True
ADD_TARGET_LOG_RETURN_VOLNORM = True

# At H=30:
#   mid_return_t_plus_30
#   target_log_return_30m
#   target_log_return_30m_volnorm
# At H=60 the corresponding names become *_60 / *_60m.

# Parallel processing
N_JOBS = -1
CHUNKS_PER_WORKER = 1
SHOW_PROGRESS = True


## Step 2 — input sanity check

The utility sorts only inside complete RIC chunks while computing temporal features and restores the original input row order afterwards.


In [ ]:
if "modeling_df" not in globals():
    raise NameError("Provide the raw input DataFrame as modeling_df before running this notebook.")

print("input shape:", modeling_df.shape)
print("rows per minute:", ROWS_PER_MINUTE)
print("target horizon minutes:", TARGET_HORIZON_MINUTES)


## Step 3 — engineer LOB features and optional targets

The same DataFrame receives the new columns. Future targets remain `NaN` where the requested horizon would cross a `(RIC, date_dt)` boundary or where the current/future book is invalid.


In [ ]:
modeling_df = add_lob_features(
    modeling_df,
    levels=LEVELS,
    group_cols=GROUP_COLS,
    time_col=TIME_COL,
    depth_ks=DEPTH_KS,
    rolling_windows=ROLLING_WINDOWS,
    volatility_minutes=VOLATILITY_MINUTES,
    rows_per_minute=ROWS_PER_MINUTE,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    add_target_mid_return=ADD_TARGET_MID_RETURN,
    add_target_log_return=ADD_TARGET_LOG_RETURN,
    add_target_log_return_volnorm=ADD_TARGET_LOG_RETURN_VOLNORM,
    distance_decay=DISTANCE_DECAY,
    n_jobs=N_JOBS,
    chunks_per_worker=CHUNKS_PER_WORKER,
    show_progress=SHOW_PROGRESS,
)

print("output shape:", modeling_df.shape)


## Step 4 — inspect volatility features and targets


In [ ]:
check_cols = [
    f"past_vol_{m}m" for m in VOLATILITY_MINUTES
    if f"past_vol_{m}m" in modeling_df.columns
]

H = TARGET_HORIZON_MINUTES
if ADD_TARGET_MID_RETURN:
    check_cols.append(f"mid_return_t_plus_{H}")
if ADD_TARGET_LOG_RETURN:
    check_cols.append(f"target_log_return_{H}m")
if ADD_TARGET_LOG_RETURN_VOLNORM:
    check_cols.append(f"target_log_return_{H}m_volnorm")

check_cols = [c for c in check_cols if c in modeling_df.columns]
display(modeling_df[check_cols].describe(percentiles=[0.01, 0.5, 0.99]).T)


## Final output

Use `modeling_df` as the feature-engineering output. The post-processing notebook can take this DataFrame as its next input.
